# MP2 OSS Activity Analysis: Part 1 (Data Retrieval)
**Author (UTK ID):** jzr266

This notebook is my copy of the provided getinfo.ipynb, adapted to retrieve commit data from World of Code (WoC) for my 10 assigned projects and to report the required GitHub and WoC statistics.

### My 10 assigned projects (from net2prj.csv)
| # | WoC project id | GitHub repository |
|---|---|---|
| 1 | panoptes_panoptes-utils | https://github.com/panoptes/panoptes-utils |
| 2 | photrek_nonlinear-statistical-coupling | https://github.com/Photrek/Nonlinear-Statistical-Coupling |
| 3 | npellet_visualizer | https://github.com/NPellet/visualizer |
| 4 | darioizzo_audi | https://github.com/darioizzo/audi |
| 5 | milvus-io_milvus | https://github.com/milvus-io/milvus |
| 6 | antelopeusersgroup_antelope_contrib | https://github.com/antelopeusersgroup/antelope_contrib |
| 7 | oceandatatools_openvdm | https://github.com/OceanDataTools/openvdm |
| 8 | fourierflows_fourierflows.jl | https://github.com/FourierFlows/FourierFlows.jl |
| 9 | gnina_libmolgrid | https://github.com/gnina/libmolgrid |
| 10 | unidata_awips2 | https://github.com/Unidata/awips2 |


In [ ]:
!python -m pip install -U python-woc pandas matplotlib tqdm

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
import time

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

projects = [
    'panoptes/panoptes-utils',
    'Photrek/Nonlinear-Statistical-Coupling',
    'NPellet/visualizer',
    'darioizzo/audi',
    'milvus-io/milvus',
    'antelopeusersgroup/antelope_contrib',
    'OceanDataTools/openvdm',
    'FourierFlows/FourierFlows.jl',
    'gnina/libmolgrid',
    'Unidata/awips2',
]

## Step 1: get all commit sha1's for each project (p2c)

In [ ]:
list_df_commits = []
for prj in projects:
    prj = prj.lower().replace('/', '_', 2)
    commits = woc.get_values('p2c', prj)
    df = pd.DataFrame(commits, columns=["sha1"])
    df['project'] = prj
    print(prj, len(df), 'commits')
    list_df_commits.append(df)

df = pd.concat(list_df_commits, ignore_index=True)
df.to_csv('df_commits.csv', index=False)
df.head(1)

## Step 2: retrieve commit content in batches (commit.tch)

The WoC batch API takes up to 10 keys at a time, with a short wait between requests.

In [ ]:
commit_data = []
for prj, g in df.groupby('project'):
    shas = g['sha1'].tolist()
    chunks = [shas[x:x+10] for x in range(0, len(shas), 10)]
    for chunk in tqdm(chunks, desc=prj):
        res, err = woc.get_values_many('commit.tch', chunk)
        res = {k: v[0] for k, v in res.items()}
        time.sleep(1)
        if err:
            print('Got Errors', err)
        for commit_sha, commit in res.items():
            commit_data.append({
                'project': prj,
                'commit':  commit_sha,
                'author':  commit[2][0],
                'time':    int(commit[2][1]),
                'message': commit[4],
            })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data.to_csv('df_commit_data.csv', index=False)
print(len(df_commit_data), 'commits retrieved')
df_commit_data.head(2)

## Step 3: save jzr266_project_summary.csv

Semicolon separated, columns: project_wocid ; commit_sha1 ; author ; time ; commit message.

In [ ]:
dfinf = df_commit_data.rename(columns={
    'project': 'project_wocid', 'commit': 'commit_sha1', 'message': 'commit message'})
dfinf = dfinf[['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']]
dfinf.to_csv('jzr266_project_summary.csv', index=False, sep=';', header=False)
print('wrote jzr266_project_summary.csv:', len(dfinf), 'rows')
dfinf.head(1)

## GitHub data (per project)

Number of stars, number of forks, and the date of the most recent commit, from each project's GitHub repository. Snapshot taken 2026-09-28.

| Project | Stars | Forks | Last commit date |
|---|---:|---:|---|
| panoptes/panoptes-utils | 4 | 8 | 2026-07-29 |
| Photrek/Nonlinear-Statistical-Coupling | 5 | 1 | 2025-11-20 |
| NPellet/visualizer | 53 | 24 | 2026-06-26 |
| darioizzo/audi | 82 | 19 | 2026-09-28 |
| milvus-io/milvus | 46,276 | 4,275 | 2026-09-29 |
| antelopeusersgroup/antelope_contrib | 54 | 62 | 2026-09-01 |
| OceanDataTools/openvdm | 15 | 6 | 2026-09-19 |
| FourierFlows/FourierFlows.jl | 218 | 32 | 2026-06-29 |
| gnina/libmolgrid | 156 | 59 | 2025-11-03 |
| Unidata/awips2 | 227 | 84 | 2026-09-25 |


## WoC data (per project)

Number of commits, number of authors, and the max and min commit time, from the commits retrieved above.

| Project | ncommits | nauthors | min time | max time |
|---|---:|---:|---|---|
| panoptes_panoptes-utils | 2,741 | 15 | 2019-03-25 | 2025-10-21 |
| photrek_nonlinear-statistical-coupling | 349 | 11 | 2020-07-24 | 2025-10-02 |
| npellet_visualizer | 7,575 | 41 | 2012-01-01 | 2025-11-05 |
| darioizzo_audi | 946 | 14 | 2015-09-01 | 2026-04-04 |
| milvus-io_milvus | 85,309 | 909 | 2019-03-18 | 2025-11-01 |
| antelopeusersgroup_antelope_contrib | 7,129 | 85 | 1997-04-12 | 2025-10-06 |
| oceandatatools_openvdm | 1,181 | 9 | 2021-02-13 | 2025-10-26 |
| fourierflows_fourierflows.jl | 2,234 | 28 | 2017-07-01 | 2025-11-02 |
| gnina_libmolgrid | 529 | 15 | 2018-12-19 | 2026-04-05 |
| unidata_awips2 | 146,004 | 159 | 2012-01-06 | 2026-04-07 |

The cell below recomputes these directly from the retrieved data.

In [ ]:
s = df_commit_data.groupby('project').agg(
        ncommits=('commit', 'nunique'),
        nauthors=('author', 'nunique'),
        min_time=('time', 'min'),
        max_time=('time', 'max'))
s['min_time'] = pd.to_datetime(s['min_time'], unit='s').dt.strftime('%Y-%m-%d')
s['max_time'] = pd.to_datetime(s['max_time'], unit='s').dt.strftime('%Y-%m-%d')
s

Check in both the notebook and jzr266_project_summary.csv to the fork.